# Config (Postgres + Ollama)

In [1]:
import os, re, json, uuid
from pathlib import Path
import requests
from tqdm import tqdm

# ===== Ollama (Phi-4)
OLLAMA_BASE = "http://localhost:11434"
OLLAMA_MODEL = "phi4"

def ollama_chat(messages, model=OLLAMA_MODEL, temperature=0.2, stream=False):
    url = f"{OLLAMA_BASE}/api/chat"
    payload = {
        "model": model,
        "messages": messages,
        "options": {"temperature": float(temperature)},
        "stream": bool(stream),
    }
    r = requests.post(url, json=payload, timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]

# ===== Postgres (ajuste para o seu ambiente)
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB", "professor_in_pocket")
PG_USER = os.getenv("PG_USER", "postgres")
PG_PASS = os.getenv("PG_PASS", "postgres")

# Dimensão do embedding (bge-m3 = 1024)
EMB_DIM = 1024

# Recuperação top-k usando pgvector (<=> cosine distance)

In [ ]:
def retrieve_pgvector(query: str, k=5, disciplina="programacao", probes=10):
    q_emb = embedder.encode([query], normalize_embeddings=True)[0].tolist()

    SQL = f"""
    SET ivfflat.probes = {int(probes)};

    SELECT
      source,
      chunk_index,
      content,
      (embedding <=> %s::vector) AS distance
    FROM pip_chunks
    WHERE disciplina = %s
    ORDER BY embedding <=> %s::vector
    LIMIT {int(k)};
    """

    with pg_connect() as conn:
        with conn.cursor(row_factory=dict_row) as cur:
            cur.execute(SQL, (q_emb, disciplina, q_emb))
            rows = cur.fetchall()

    hits = []
    for r in rows:
        hits.append({
            "source": r["source"],
            "chunk_index": r["chunk_index"],
            "text": r["content"],
            "distance": float(r["distance"]),
        })
    return hits

def format_sources(hits, max_chars_each=450):
    lines = []
    for i, h in enumerate(hits, 1):
        excerpt = h["text"][:max_chars_each].replace("\n", " ").strip()
        lines.append(f"[{i}] {h['source']} (chunk {h['chunk_index']})\n    Trecho: {excerpt}...")
    return "\n".join(lines)

test_hits = retrieve_pgvector("Qual é o objetivo do Professor in Pocket?", k=4, probes=10)
print(format_sources(test_hits))

# Resposta com Phi-4 (RAG + fontes)

In [ ]:
SYSTEM_RULES = """Você é um tutor acadêmico.
Regras obrigatórias:
1) Use SOMENTE o contexto fornecido (material oficial).
2) Se a resposta não estiver no contexto, diga claramente: "Não encontrei base suficiente no material fornecido."
3) Sempre inclua uma seção "Fontes" citando [1], [2], etc.
4) Seja direto e didático, sem inventar detalhes.
"""

def answer_rag_pgvector(query: str, k=5, probes=10, disciplina="programacao"):
    hits = retrieve_pgvector(query, k=k, probes=probes, disciplina=disciplina)
    context = "\n\n".join([f"[{i+1}] {h['text']}" for i, h in enumerate(hits)])
    sources = format_sources(hits)

    user_prompt = f"""Pergunta: {query}

Contexto (trechos do material):
{context}

Agora responda seguindo as regras. No final, traga "Fontes" referenciando os índices [1], [2]...
"""
    messages = [
        {"role": "system", "content": SYSTEM_RULES},
        {"role": "user", "content": user_prompt},
    ]
    resp = ollama_chat(messages, temperature=0.2)
    resp += "\n\n---\nFontes (auditável):\n" + sources
    return resp

print(answer_rag_pgvector("O que é o Professor in Pocket e qual o objetivo central?", k=5, probes=10)[:2500])

# Testes de validação

In [ ]:
tests = [
    "O que é o Professor in Pocket e qual o objetivo central?",
    "Como funciona a camada de orquestração e o classificador?",
    "Quais são as fases do piloto e o cronograma estimado?",
    "Qual é a cor favorita do reitor? (isso não está no material)",
]

for q in tests:
    print("\n" + "="*90)
    print("Q:", q)
    print(answer_rag_pgvector(q, k=5, probes=10)[:1800])